# Stage 1: Data Understanding & Exploratory Data Analysis (EDA)
## PAN Authorship Attribution & Generative AI Detection (Days 1–2)

This notebook implements and documents all requirements for **Punto 1** of the project:
1. **Literature Review & Theoretical Framing (Point 11)**: Stylometric feature analysis and literature comparison matrix.
2. **Pipeline Architecture**: End-to-end processing flowchart.
3. **Initial Inspection (Points 1, 2, 8)**: Extraction of dataset size ($), classes ($), and sample row exploration.
4. **Data Quality & Balance (Points 3, 4, 12)**: Programmatic missing values, integrity audit, and class balance bar charts.
5. **Structural & Stylometric Distribution Analysis (Points 10, 11)**: Text length variances, Type-Token Ratio (TTR), punctuation density, and n-grams.


---
### 1. Literature Review & Methodology (Point 11)
#### 1.1 Stylometric Feature Taxonomy in Authorship Attribution
Stylometry leverages quantifiable linguistics markers that reflect an author's unconscious habits:
- **Lexical Richness**: Quantified via *Type-Token Ratio* ( = V / N$) or *Yule’s Characteristic , measuring vocabulary variation.
- **Syntactic & Punctuation Markers**: Distribution and density of punctuation marks (), clause boundaries, and sentence lengths.
- **Function Words / Stop-word Frequencies**: Frequency of context-independent words (pronouns, prepositions, conjunctions) which remain invariant to topic shifts.
- **Character & Word N-grams**: Sub-word character n-grams ( \in [3, 5]$) capture morphological preferences, prefixes, suffixes, and punctuation collocations.

#### 1.2 PAN Literature Comparison Matrix
| Approach / Paper | Core Features | Classification Model | Target Task | Key Limitation |
| :--- | :--- | :--- | :--- | :--- |
| **Stamatatos (2009, 2013)** | Character 3-grams, stop-word profiles | SVM / Ensemble Linear Classifiers | Traditional Authorship Identification | High dimensionality; topic sensitivity |
| **Koppel et al. (2011)** | 'Unmasking' technique, chunk degradation | Linear SVM iterations | Authorship Verification (1 vs 1) | Requires long texts; computationally slow |
| **Barlas & Stamatatos (2020)** | Pretrained Transformers (BERT, RoBERTa) | Fine-tuned Neural Cross-Entropy | Cross-domain Authorship Attribution | Black-box; prone to topic overfitting |
| **Our Proposed Pipeline** | Multi-level Stylometrics (TTR, Punctuation, Char/Word N-grams, Length Variance) | Scalable Linear/Tree Ensemble (Baseline) + Fine-tuning | PAN 2025 GenAI Detection & Attribution | Balances interpretability with high predictive power |


---
### 2. Environment Setup & Core Module Imports


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Set project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_utils import (
    find_default_train_path,
    find_default_val_path,
    inspect_dataset_structure,
    present_sample_records,
    evaluate_data_quality,
    extract_exploratory_features,
)
from src.visualization import (
    plot_binary_class_balance,
    plot_author_model_distribution,
    plot_genre_distribution,
    plot_text_length_distribution,
    plot_stylometric_distributions,
    plot_top_ngrams,
)

train_path = find_default_train_path()
val_path = find_default_val_path()
figures_dir = ROOT / "reports" / "figures"
print(f"Training Dataset Path: {train_path}")
print(f"Validation Dataset Path: {val_path}")


---
### 3. Initial Inspection (Points 1, 2, 8)
Extracting total number of records ($), binary classes ({label}$), candidate authors / generative models ({model}$), and domain genres ({genre}$).

In [ ]:
# Compute dataset dimensions
train_stats = inspect_dataset_structure(train_path)
val_stats = inspect_dataset_structure(val_path) if val_path else {}

print("=== DATASET STRUCTURAL INSPECTION ===")
print(f"Total Training Records (N_train): {train_stats['total_records_N']:,}")
if val_stats:
    print(f"Total Validation Records (N_val): {val_stats['total_records_N']:,}")
    print(f"Grand Total Records (N_total):    {train_stats['total_records_N'] + val_stats['total_records_N']:,}")
print(f"Binary Target Classes (k_labels):   {train_stats['k_labels']} -> {train_stats['label_distribution']}")
print(f"Candidate Authors / Models (k):     {train_stats['k_models_authors']}")
print(f"Domains / Genres (k_genres):        {train_stats['k_genres']} -> {train_stats['genre_distribution']}")


#### Sample Row Inspection (Points 1, 2, 8)
Inspecting schema fields and raw text snippets.

In [ ]:
samples = present_sample_records(train_path, n_samples=3)
for s in samples:
    print(f"Sample #{s['sample_index']}:")
    print(f"  • ID:           {s['id']}")
    print(f"  • Author/Model: {s['author_model']}")
    print(f"  • Class:        {s['label_name']}")
    print(f"  • Genre:        {s['genre']}")
    print(f"  • Dimensions:   {s['total_words']} words, {s['total_chars']} chars")
    print(f"  • Text Snippet: "{s['text_preview']}"
")


---
### 4. Data Quality & Balance Audit (Points 3, 4, 12)
Programmatic verification of data integrity, corrupted records, missing values, duplicate IDs, and text duplicates.

In [ ]:
dq = evaluate_data_quality(train_path)
print("=== DATA QUALITY AUDIT RESULTS ===")
print(f"• Lines Processed:      {dq['total_lines_read']:,} (Parsed: {dq['total_records_parsed']:,})")
print(f"• Corrupted JSON Lines: {dq['corrupted_lines']}")
print(f"• Missing Schema Fields:{dq['missing_fields']}")
print(f"• Empty / Null Texts:   {dq['empty_text_count']}")
print(f"• Non-String Texts:     {dq['non_string_text_count']}")
print(f"• Duplicate IDs:        {dq['duplicate_id_count']}")
print(f"• Duplicate Texts:      {dq['duplicate_text_count']}")
print(f"• Length Extreme Range: [{dq['min_word_count']} - {dq['max_word_count']}] words (Mean: {dq['mean_word_count']})")
print(f"• Integrity Status:     {dq['data_integrity_status']}")


---
### 5. Class Balance & Distribution Charts (Points 4, 12)
Visualizing binary balance, candidate authors / models, and genre representation.

In [ ]:
display(Image(filename=str(figures_dir / "class_balance_binary.png")))
display(Image(filename=str(figures_dir / "class_balance_authors.png")))
display(Image(filename=str(figures_dir / "class_balance_genres.png")))


---
### 6. Text Length Distributions & Structural Variance (Point 10)
Evaluating word count and character count discrepancies between Human and AI texts.

In [ ]:
display(Image(filename=str(figures_dir / "text_length_distribution.png")))


---
### 7. Stylometric Feature Density Distributions (Points 10, 11)
Comparing Type-Token Ratio (Lexical Richness), Punctuation Density, and Average Word Length across classes.

In [ ]:
display(Image(filename=str(figures_dir / "stylometric_distributions.png")))


---
### 8. N-gram Analysis (Top Unigrams and Bigrams)
Horizontal bar charts illustrating lexical collocations and content token signatures.

In [ ]:
display(Image(filename=str(figures_dir / "top_ngrams_comparison.png")))


---
### 9. Feature Distributions: Entire Dataset & Author-Level Breakdown (Point 10)
Histograms with continuous KDE curves and comparative boxplots showing the distributions of key stylometric features ($):
1. **Across the Entire Dataset**: Overall shape, skewness, median, and IQR across all =23,707$ texts.
2. **Among Different Candidate Authors / LLM Models**: Scomposizione comparativa tra l'Autore Umano e i principali modelli generativi (GPT-4o, Gemini, LLaMA, Ministral, DeepSeek).

In [ ]:
# 1. Feature Distributions Across Entire Dataset (Point 10)
display(Image(filename=str(figures_dir / "feature_distributions_entire_dataset.png")))

In [ ]:
# 2. Feature Distributions Among Different Candidate Authors / Models (Point 10)
display(Image(filename=str(figures_dir / "feature_distributions_by_author.png")))

---
### 10. Correlation Matrix, Collinearity & Predictive Power (Points 9, 13)
Empirical Pearson correlation analysis on the full training dataset (=23,707$):
1. **Heatmap of Inter-Feature Collinearity**: Identifies redundant variables measuring the same underlying construct ($|r| \ge 0.60$).
2. **Feature Predictive Power Regarding Output ($)**: Ranks features by their discriminative strength between Human (=0$) and AI-Generated (=1$) texts.

In [ ]:
# 1. Correlation Matrix Heatmap (Points 9, 13)
display(Image(filename=str(figures_dir / "correlation_heatmap.png")))

In [ ]:
# 2. Predictive Power Ranking Regarding Output (Points 9, 13)
display(Image(filename=str(figures_dir / "feature_predictive_power_ranking.png")))

---
### 11. Key Insights & Preprocessing Roadmap (Days 3–4)
1. **Class Disparity**: Training set comprises 61.6% AI-generated texts across 22 architectures and 38.4% Human texts.
2. **Length Discrepancy**: Human texts are noticeably longer on average (710.9 words) than AI texts (584.5 words), indicating a systematic structural marker.
3. **Punctuation Bias**: Human authors exhibit ~32% higher punctuation density (zsh.0375$ vs zsh.0284$), reflecting more complex clause structures, parentheticals, and dialogue markers.
4. **Lexical Richness (TTR)**: AI models exhibit higher Type-Token Ratio (zsh.5224$ vs zsh.4714$), consistent with modern LLM decoders optimizing for vocabulary diversity.
5. **Action Plan for Days 3–4**: Define tokenization and normalization pipelines, extract sub-word character n-grams to mitigate length bias, and engineer custom stylometric ratios (Point 13).